# Assumption checking short use cases

Some assumptions cannot be tested statistically from the data alone. For these, we use an LLM to reason about their plausibility based on the dataset description and domain knowledge.

Set your `OPENAI_API_KEY` in the environment before running the cells below.

In [ ]:
from cais.config import get_llm_client

try:
    llm = get_llm_client()
    print(f"LLM client ready: {type(llm).__name__}")
except Exception as e:
    llm = None
    print(f"No LLM available — LLM-based checks will return passed=None. ({e})")

## Pre-modeling assumptions

#### Cond-ignorability (IPW, matching)

In [ ]:
import numpy as np
import pandas as pd
from cais.models import AssumptionVariables
from cais.methods.pre_model_assumption_utils import check_cond_ignorability, check_sutva, check_positivity
from cais.methods.propensity_score.base import estimate_propensity_scores

In [ ]:
np.random.seed(42)
n = 500
age    = np.random.normal(40, 10, n)
income = np.random.normal(50_000, 15_000, n)
# Confounded: older and higher-income individuals more likely to be treated
ps_true = 1 / (1 + np.exp(-(-2 + 0.05 * age + 0.00002 * income)))
treat  = (np.random.uniform(size=n) < ps_true).astype(int)
y      = 2 * treat + 0.05 * age + np.random.normal(0, 1, n)

df_psm = pd.DataFrame({"treat": treat, "outcome": y, "age": age, "income": income})
covariates_psm = ["age", "income"]

vars_psm = AssumptionVariables(
    df=df_psm, treatment="treat", outcome="outcome", covariates=covariates_psm,
    dataset_description=(
        "Observational study of a job training program (n=500). "
        "Treatment is binary (1 = received training). Covariates: age and income. "
        "Older and higher-income individuals are more likely to receive training."
    ),
    variables_summary={"treatment": "treat", "outcome": "outcome", "covariates": covariates_psm},
)

result_cond_ignorability = check_cond_ignorability(vars_psm)
print(result_cond_ignorability)

In [ ]:
ps = estimate_propensity_scores(df_psm, "treat", covariates_psm)

In [ ]:
result_positivity = check_positivity(vars_psm, propensity_scores=ps)
print(result_positivity)

In [ ]:
result_sutva = check_sutva(vars_psm, llm=llm)
print(result_sutva)

#### Instrumental Variables (IVs)

In [ ]:
from cais.methods.pre_model_assumption_utils import (
    check_iv_relevance, check_iv_exclusion, check_iv_exogeneity, check_iv_monotonicity,
)

In [ ]:
np.random.seed(7)
n_iv = 600
nearc4  = np.random.binomial(1, 0.5, n_iv)
educ    = 10 + 2.5 * nearc4 + np.random.normal(0, 1.5, n_iv)
lwage   = 5.5 + 0.1 * educ + np.random.normal(0, 0.5, n_iv)

df_iv = pd.DataFrame({
    "nearc4": nearc4, "educ": educ, "lwage": lwage,
    "black":   np.random.binomial(1, 0.3, n_iv),
    "smsa":    np.random.binomial(1, 0.6, n_iv),
    "south":   np.random.binomial(1, 0.4, n_iv),
    "married": np.random.binomial(1, 0.5, n_iv),
    "exper":   np.random.randint(1, 25, n_iv).astype(float),
})
iv_covariates = ["black", "smsa", "south", "married", "exper"]

In [ ]:
print(f"IV dataset — {df_iv.shape[0]} obs, {df_iv.shape[1]} columns")
df_iv.head()

In [ ]:
vars_iv = AssumptionVariables(
    df=df_iv, treatment="educ", outcome="lwage",
    instruments=["nearc4"], covariates=iv_covariates,
)

In [ ]:
result_iv_relevance = check_iv_relevance(vars_iv, f_threshold=10.0)
print(result_iv_relevance)

In [ ]:
card_description = (
    "Card (1995) — returns to education. Treatment: years of education (educ). "
    "Instrument: nearc4 (grew up near a 4-year college). Outcome: log wage (lwage). "
    "The exclusion restriction argument: college proximity affects education but not "
    "wages directly — though this is debated, since proximity may correlate with "
    "local labour market conditions."
)
card_variables = {
    "treatment": "educ", "outcome": "lwage",
    "instrument": "nearc4", "covariates": iv_covariates,
}

vars_iv_llm = AssumptionVariables(
    df=df_iv, treatment="educ", outcome="lwage",
    instruments=["nearc4"], covariates=iv_covariates,
    dataset_description=card_description, variables_summary=card_variables,
)

result_iv_exclusion = check_iv_exclusion(vars_iv_llm, llm=llm)
print("check_iv_exclusion :", result_iv_exclusion)

In [ ]:
result_iv_exogeneity = check_iv_exogeneity(vars_iv_llm, llm=llm)
print("check_iv_exogeneity :", result_iv_exogeneity)

In [ ]:
result_iv_monotonicity = check_iv_monotonicity(vars_iv_llm, llm=llm)
print("check_iv_monotonicity :", result_iv_monotonicity)

In the description furnished before, there wasn't enough description for the LLM to judge whether IV monocity was valid. That's why it returns `'None'`.

We enrich the description now:

In [ ]:
card_description_enriched = (
    "Card (1995) dataset. Treatment: years of education. Instrument: nearc4. "
    "The instrument works through reduced cost of attending college: individuals near "
    "a college face lower transportation and housing costs, making them more likely to "
    "attend. It is implausible that proximity would cause someone to get LESS education "
    "— the effect goes in one direction only (proximity → more education or no change)."
)

vars_iv_enriched = AssumptionVariables(
    df=df_iv, treatment="educ", outcome="lwage",
    instruments=["nearc4"], covariates=iv_covariates,
    dataset_description=card_description_enriched, variables_summary=card_variables,
)

In [ ]:
result_iv_monotonicity_enriched = check_iv_monotonicity(vars_iv_enriched, llm=llm)
print("check_iv_monotonicity (enriched description):", result_iv_monotonicity_enriched)

#### Difference-in-Differences (DiD)

In [ ]:
from cais.methods.pre_model_assumption_utils import (
    check_parallel_trends, check_no_anticipation,
    check_baseline_outcome_balance, check_stable_group_composition,
)

In [ ]:
np.random.seed(0)
n_states, n_periods, treatment_year = 50, 8, 5
rows = []
for sid in range(n_states):
    ever_treated = int(sid >= 25)  # states 25-49 are treated
    for t in range(n_periods):
        y = (ever_treated * 0.5          # baseline level difference
             + t * 1.0                   # common time trend (parallel by design)
             + ever_treated * (t >= treatment_year) * 2.5  # true treatment effect
             + np.random.normal(0, 0.5))
        rows.append({
            "sid": sid, "year": t, "ever_treated": ever_treated,
            "l_homicide": y,
            "l_police": np.random.normal(3, 0.3),
            "l_income": np.random.normal(10, 0.5),
        })

df_did = pd.DataFrame(rows)
covariates_did = ["l_police", "l_income"]

vars_did = AssumptionVariables(
    df=df_did, treatment="ever_treated", outcome="l_homicide",
    time_var="year", group_var="sid",
    treatment_period_start=treatment_year, placebo_period_start=2,
    covariates=covariates_did,
    dataset_description=(
        "Panel of 50 states, 8 periods. Treatment: adoption of Castle Doctrine laws "
        "(states 25-49). Outcome: log homicide rate. Parallel trends hold by design."
    ),
    variables_summary={"treatment": "ever_treated", "outcome": "l_homicide",
                       "time": "year", "panel_id": "sid"},
)

df_did.head(10)

In [ ]:
result_parallel_trends = check_parallel_trends(vars_did)
print(result_parallel_trends)

In [ ]:
result_no_anticipation = check_no_anticipation(vars_did)
print(result_no_anticipation)

In [ ]:
result_baseline_balance = check_baseline_outcome_balance(vars_did)
print(result_baseline_balance)

In [ ]:
result_stable_group = check_stable_group_composition(vars_did, llm=llm)
print(result_stable_group)

In [ ]:
# Enrich the description with explicit panel completeness info
vars_did_enriched = AssumptionVariables(
    df=df_did, treatment="ever_treated", outcome="l_homicide",
    time_var="year", group_var="sid",
    treatment_period_start=treatment_year,
    dataset_description=(
        "Panel of 50 US states, 8 balanced periods. All 50 states are observed for all "
        "8 periods — no missing state-period observations. States do not enter or exit. "
        "There is no differential attrition due to treatment."
    ),
    variables_summary=vars_did.variables_summary,
)
result_stable_group_enriched = check_stable_group_composition(vars_did_enriched, llm=llm)
print(result_stable_group_enriched)

In [ ]:
result_sutva_did = check_sutva(vars_did, llm=llm)
print(result_sutva_did)

#### Frontdoor adjustment

In [ ]:
from cais.methods.pre_model_assumption_utils import (
    check_frontdoor_full_mediation, check_frontdoor_no_TM_confounding,
    check_frontdoor_T_blocks_MY, check_frontdoor_positivity,
)

In [ ]:
np.random.seed(10)
n_fd = 400
T_fd = np.random.binomial(1, 0.4, n_fd)
M_fd = np.random.binomial(1, 0.3 + 0.5 * T_fd)
df_fd = pd.DataFrame({"T": T_fd, "M": M_fd, "Y": M_fd * 0.6 + np.random.normal(0, 0.3, n_fd)})

frontdoor_description = (
    "Observational study: effect of smoking (T) on lung cancer (Y). "
    "Tar deposits in lungs (M) are the mediator. "
    "There may be unobserved genetic confounders affecting both T and Y."
)
frontdoor_variables = {
    "treatment": "smoking (T)", "mediator": "tar deposits (M)",
    "outcome": "lung cancer (Y)", "potential_confounders": "genetic predisposition (unobserved)",
}

vars_fd = AssumptionVariables(
    df=df_fd, treatment="T", outcome="Y", mediator="M",
    dataset_description=frontdoor_description, variables_summary=frontdoor_variables,
)

In [ ]:
result_full_mediation = check_frontdoor_full_mediation(vars_fd, llm=llm)
print("check_frontdoor_full_mediation:", result_full_mediation)

In [ ]:
result_no_tm_conf = check_frontdoor_no_TM_confounding(vars_fd, llm=llm)
print("check_frontdoor_no_TM_confounding:", result_no_tm_conf)

In [ ]:
result_t_blocks = check_frontdoor_T_blocks_MY(vars_fd, llm=llm)
print("check_frontdoor_T_blocks_MY:", result_t_blocks)

In [ ]:
result_fd_positivity = check_frontdoor_positivity(vars_fd, min_count=5)
print("Normal case:", result_fd_positivity)

In [ ]:
df_fd_bad = df_fd[~((df_fd["T"] == 1) & (df_fd["M"] == 0))].copy()
vars_fd_bad = AssumptionVariables(df=df_fd_bad, treatment="T", outcome="Y", mediator="M")
result_fd_positivity_bad = check_frontdoor_positivity(vars_fd_bad, min_count=5)
print("Violation case:", result_fd_positivity_bad)

#### Regression Discontinuity Design (RDD)

In [ ]:
from cais.methods.pre_model_assumption_utils import (
    check_rdd_no_manipulation, check_rdd_covariate_continuity,
    check_rdd_continuity_potential_outcomes,
)

In [ ]:
np.random.seed(11)
n_rdd = 3000
# Running variable symmetric around cutoff 0.08 → no natural density imbalance
bac1 = np.random.uniform(0.00, 0.16, n_rdd)
treated_rdd = (bac1 >= 0.08).astype(int)
df_rdd = pd.DataFrame({
    "bac1":       bac1,
    "male":       np.random.binomial(1, 0.7, n_rdd),
    "white":      np.random.binomial(1, 0.6, n_rdd),
    "acc":        np.random.binomial(1, 0.1, n_rdd),
    "aged":       np.random.normal(30, 8, n_rdd),
    "recidivism": 0.3 - 0.1 * treated_rdd + np.random.normal(0, 0.2, n_rdd),
})

rdd_description = (
    "Synthetic DUI records. Running variable: BAC (blood alcohol content). "
    "Cutoff: 0.08. Treatment: DUI sanction. Outcome: recidivism (future DUI). "
    "Individuals cannot precisely control their BAC during a breathalyzer test."
)

print(f"Shape: {df_rdd.shape}  |  BAC range: {df_rdd['bac1'].min():.3f}-{df_rdd['bac1'].max():.3f}")
print(f"Treated (BAC >= 0.08): {treated_rdd.sum()} / {n_rdd}")

In [ ]:
vars_rdd = AssumptionVariables(
    df=df_rdd, running_variable="bac1", cutoff=0.08,
    covariates=["male", "white", "acc", "aged"],
    dataset_description=rdd_description,
    variables_summary={"running_variable": "bac1", "cutoff": 0.08},
)

result_no_manip = check_rdd_no_manipulation(vars_rdd)
print("check_rdd_no_manipulation:", result_no_manip)

In [ ]:
result_cov_cont = check_rdd_covariate_continuity(vars_rdd)
print("check_rdd_covariate_continuity:", result_cov_cont)

In [ ]:
result_po_continuity = check_rdd_continuity_potential_outcomes(vars_rdd, llm=llm)
print("check_rdd_continuity_potential_outcomes:", result_po_continuity)

## Post-modeling assumptions

#### Generalized Propensity Score

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from cais.methods.post_model_assumption_utils import (
    check_gps_specification,
    check_balance_after_matching,
    check_balance_after_weighting,
    check_iv_overidentification,
)

Let's compute Generalized Propensity Scores on synthetic continuous-treatment data and verify residual normality of the GPS model.

In [ ]:
np.random.seed(99)
n_gps = 300
X1_gps = np.random.normal(0, 1, n_gps)
X2_gps = np.random.binomial(1, 0.5, n_gps)
T_gps  = 5 + 1.5 * X1_gps - 0.5 * X2_gps + np.random.normal(0, 2, n_gps)
Y_gps  = 2 * T_gps + X1_gps + np.random.normal(0, 1, n_gps)

df_gps = pd.DataFrame({"T": T_gps, "Y": Y_gps, "X1": X1_gps, "X2": X2_gps})
vars_gps = AssumptionVariables(df=df_gps, treatment="T", outcome="Y", covariates=["X1", "X2"])

print(f"Shape: {df_gps.shape}  |  T range: {T_gps.min():.2f}–{T_gps.max():.2f}")

In [ ]:
X = sm.add_constant(df_gps[["X1", "X2"]])
gps_model = sm.OLS(df_gps["T"], X).fit()
residuals = gps_model.resid.values
print(f"Residuals: n={len(residuals)}, mean={residuals.mean():.4f}, std={residuals.std():.4f}")

In [ ]:
result_gps_spec = check_gps_specification(residuals)
print("check_gps_specification (synthetic):", result_gps_spec)

In [ ]:
residuals_normal = np.random.default_rng(15).normal(0, 1, 300)
result_gps_normal = check_gps_specification(residuals_normal)
print("check_gps_specification (normal parfait):", result_gps_normal)

In [ ]:
residuals_bad = np.concatenate([np.random.exponential(2, 200), -np.random.exponential(2, 100)])
result_gps_bad = check_gps_specification(residuals_bad)
print("check_gps_specification (non-normal):", result_gps_bad)

#### Balance checks (IPW, matching)

In [ ]:
np.random.seed(20)
n_m = 400
age_m    = np.random.normal(40, 10, n_m)
income_m = np.random.normal(50_000, 15_000, n_m)
ps_conf  = 1 / (1 + np.exp(-(-1.5 + 0.04 * age_m + 0.00001 * income_m)))
treat_m  = (np.random.uniform(size=n_m) < ps_conf).astype(int)
y_m      = 3 * treat_m + 0.02 * age_m + np.random.normal(0, 1, n_m)
re74_m   = np.random.normal(15_000, 5_000, n_m)
re75_m   = re74_m + np.random.normal(500, 1_000, n_m)

df_m = pd.DataFrame({
    "treat": treat_m, "outcome": y_m,
    "age": age_m, "income": income_m, "re74": re74_m, "re75": re75_m,
})
covariates_m = ["age", "income", "re74", "re75"]
vars_m = AssumptionVariables(df=df_m, treatment="treat", outcome="outcome", covariates=covariates_m)

print(f"Dataset: {len(df_m)} obs, treated={treat_m.sum()}, controls={n_m - treat_m.sum()}")

Nearest-neighbour matching on propensity scores, then check covariate balance in the matched sample.

In [ ]:
ps_m = LogisticRegression(max_iter=1000).fit(df_m[covariates_m], treat_m).predict_proba(df_m[covariates_m])[:, 1]
ps_m = np.clip(ps_m, 0.01, 0.99)

treated_idx_m = df_m[treat_m == 1].index
control_idx_m = df_m[treat_m == 0].index

nn = NearestNeighbors(n_neighbors=1)
nn.fit(ps_m[treat_m == 0].reshape(-1, 1))
_, indices = nn.kneighbors(ps_m[treat_m == 1].reshape(-1, 1))
matched_ctrl_idx = control_idx_m[indices.flatten()]

df_matched_m = pd.concat([df_m.loc[treated_idx_m], df_m.loc[matched_ctrl_idx]])
print(f"Matched sample: {len(df_matched_m)} rows ({treat_m.sum()} treated + {treat_m.sum()} matched controls)")

In [ ]:
result_balance_match = check_balance_after_matching(vars_m, df_matched=df_matched_m)
print("check_balance_after_matching:", result_balance_match)

Now compute IPW weights and check weighted covariate balance on the same dataset.

In [ ]:
weights_m = np.where(treat_m == 1, 1 / ps_m, 1 / (1 - ps_m))
result_balance_ipw = check_balance_after_weighting(vars_m, weights=weights_m)
print("check_balance_after_weighting:", result_balance_ipw)

#### IV overidentification test (post-estimation)

In [ ]:
# Single instrument → test not applicable (need more instruments than endogenous regressors)
vars_iv_single = AssumptionVariables(
    df=df_iv, treatment="educ", outcome="lwage",
    instruments=["nearc4"], covariates=iv_covariates,
)
result_overid_single = check_iv_overidentification(vars_iv_single, sm_results=None)
print("Single instrument:", result_overid_single)

With two instruments, the Sargan-Hansen test can be performed. Note: `smsa` is added here as a second instrument for illustration only — in practice it would not satisfy the exclusion restriction.

In [ ]:
from statsmodels.sandbox.regression.gmm import IV2SLS

instruments_2   = ["nearc4", "smsa"]
covariates_iv2  = ["black", "south", "married", "exper"]

endog            = df_iv["lwage"]
exog             = sm.add_constant(df_iv[["educ"] + covariates_iv2])
instrument_matrix = sm.add_constant(df_iv[instruments_2 + covariates_iv2])

iv2sls = IV2SLS(endog, exog, instrument_matrix).fit()

In [ ]:
vars_iv2 = AssumptionVariables(
    df=df_iv, treatment="educ", outcome="lwage",
    instruments=instruments_2, covariates=covariates_iv2,
)
result_overid_two = check_iv_overidentification(vars_iv2, sm_results=iv2sls)
print("Two instruments (Sargan-Hansen):", result_overid_two)

#### Instrumental Variables (IVs)

In [ ]:
# (check_iv_overidentification is already imported above)

In [ ]:
# (shown above — dc0ecd56a3c12ad2)

In [ ]:
# (shown above — e5f107380fa76b1d)